# Experiment 3: Epistemic Foraging with Competing Drives

This experiment builds on Experiment 1 and offers a twist on the two-armed bandit example found in the pymdp repo.

Similarly to Experiment 1, we have two goals which provide relief from competing drives.

This time however, each goal could be in one of two locations with 50% probability.

There is a 'cue' square in the corners furthest away from the potential goals. Moving on to it is a move away from the goal.

If the agent lands on it, its observation removes all uncertainty about the world state.

It gets clear information about both goal locations - now it knows with 100% probability which to pick.

This is acheived by having two new hidden states (true location of each goal).

They have two possible values (Location 1 or 2).

If the agent is at any other square than the cue, it observes Null which doesn't provide any information.

If it is on the cue square, its observation includes the true locations, so it updates its state prediction to reflect that.

The A matrix has pre-encoded info
- If you aren't on Cue, expect to observe Null 'true goal location' state prediction
- If you are on Cue, expect to observe confident 'true goal information' state prediction

The B matrix has pre encoded info
- There's a 100% correlation between the two goal location states (0 or 1) and which location transitions lead to relief from hunger / stress. 
- The more confidently you know those state vars, the more confident you are of the locations that successful policies will visit.
- You can say 'rather than my starting 50 / 50 between these two candidate locations, given we are in the 'candidate 1 wins' world, then transitions to candidate 1 *will* 100% give me hunger / stress relief and candidate 2 transitions 100% end up with penalty. 
- Moving into a state where the location is not the true location but *is* in the candidate list (i.e. is the wrong choice) *will* transition to *maximum* hunger / stress on the next timestep. 
- This adds an incentive to avoid gambling as there is an actual penalty, dissuading an agent from travelling through both candidates if it was a shorter path than info. 

Param notes 

- For penalty from picking wrong location to have an effect, you need headroom in the max hunger / stress values as if both info and gamble paths ultimately end up at max stress there is no penalty either way when looking at the policy as a whole.
- For the path finding to work, the policy length between any two points in the journey needs to be the manhatten distance (shortest path without diagonal jumps) + 1, as the agent needs to imagine (B matrix) how, after landing on the goal, their corresponding drive will change at the next timestep
- Your hunger state tansition from t to t+1 is a function of
1.) Your current hunger
2.) Your location
3.) The true goal location
i.e. If not a candidate, current_value +1 / If true goal, =0 / If candidate but not true goal, =max
- This means we need a 3D B matrix for each of the hunger and state single-possible-action transitions.

Even in a 3 * 3 grid we need a policy length of 5 in order to let the agent predict the value of moving between any two points (max 4 manhatten for travel time corner to corner , +1 timestep for imagining the result of the 'drive' actions you take on arrival).

This large B matrix and long policy means it runs pretty slowly (21 turns takes just over 6 minutes). It will be interesting to see if the JAX implementation helps here.

Example transitions -
- 'If I land on the real food pile at time t, only then can I choose to eat, i.e. the 'hunger' action will cause me to transition to minimum at t+1'
- 'If I land on the fake food pile at time t, I will suffer a catastrophe and transition to max hunger at t+1'
- 'If I land on any square that isn't the food pile at time t, I will get an (increasingly disliked) hunger increment of +1 at t+1'
- 'At the start, landing on a candidate square may reset or max out my hunger depending upon the hidden goal var which we are ambiguous about, so any policy that passes through them has equal chance of win or catastrophe, I can't confidently mark the policy high or low.'
- 'If I land on the cue square, I observe the goal var for certain so I am no longer ambiguous about the state of the world and know how landing on a candidate will turn out so *can* confidently score that policy high or low'
- 'I might end up in a more hungry / stressed state by visiting the info, but crucially I won't end up as bad as if I had gambled and lost'
- 'If both policies ultimately end up in max discomfort, who cares which I choose?'

You could tweak the parameters to balance risk avoidance and get it to gamble over info seeking, depending on how much it hates small amounts of hunger and stress.